# Predykcja bankructwa polskich firm — pipeline bez wycieku danych

**Zbiór:** `5year.arff`, czyli prognoza bankructwa w następnym roku.  
**Cel:** klasyfikacja `class=1` (bankructwo).  
**Metodologia:** rozdzielenie treningu i testu przed dopasowaniem imputacji, winsoryzacji i skalowania; stratyfikowana walidacja krzyżowa na treningu; metryka główna Average Precision (PR-AUC).  
**Uwaga:** plik Excel `Dane_ML_gotowe` nie jest używany do testu, gdyż zawiera mediany i granice winsoryzacji wyznaczone na całych danych.

Kod został przygotowany dla scikit-learn ≥ 1.2 i danych lokalnych, bez połączenia z internetem.

## 1. Importy i wczytanie surowego zbioru

Nie mieszamy `1year`–`5year`: są to różne horyzonty prognozy, a bez identyfikatorów firm nie możemy wykluczyć zależności między nimi.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import arff

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_recall_curve, precision_score, recall_score,
    fbeta_score, balanced_accuracy_score, confusion_matrix,
    classification_report
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate,
    RandomizedSearchCV, cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from xgboost import XGBClassifier


RANDOM_STATE = 42
PLIK = Path("5year.arff")  

rekordy, meta = arff.loadarff(str(PLIK))
kolumny = [n for n in rekordy.dtype.names if n != "class"]
X = pd.DataFrame({n: rekordy[n].astype(float) for n in kolumny})
y = pd.Series([int(k.decode("ascii")) for k in rekordy["class"]], name="class")

# Identyczne wiersze (łącznie z etykietą) usuwamy przed podziałem.
dane = X.copy()
dane["class"] = y
print("Surowych rekordów:", len(dane))
print("Duplikaty X + y:", int(dane.duplicated().sum()))
dane = dane.drop_duplicates().reset_index(drop=True)
y = dane.pop("class").astype(int)
X = dane
print("Po usunięciu duplikatów:", X.shape)
print("Bankructwa:", int(y.sum()), "udział:", round(y.mean() * 100, 2), "%")

Surowych rekordów: 5910
Duplikaty X + y: 60
Po usunięciu duplikatów: (5850, 64)
Bankructwa: 408 udział: 6.97 %


## 2. Wstępna diagnostyka

Diagnoza służy zrozumieniu problemu. Przy wyborze hiperparametrów, cech, progów lub strategii imputacji używamy tylko danych treningowych.

In [2]:
raport_brakow = (
    X.isna().mean().mul(100).sort_values(ascending=False)
    .rename("odsetek_brakow_pct").to_frame()
)
print("Odsetek brakujących komórek:", round(X.isna().to_numpy().mean() * 100, 2), "%")
print("Przykładowe kolumny z największą liczbą braków:")
print(raport_brakow.head(10).to_string())
print("Dokładnie zduplikowane wiersze po czyszczeniu:", int(X.duplicated().sum()))
print("Liczba cech:", X.shape[1])

Odsetek brakujących komórek: 1.24 %
Przykładowe kolumny z największą liczbą braków:
        odsetek_brakow_pct
Attr37           43.162393
Attr27            6.666667
Attr45            4.547009
Attr60            4.547009
Attr24            2.307692
Attr28            1.811966
Attr64            1.811966
Attr53            1.811966
Attr54            1.811966
Attr21            1.760684
Dokładnie zduplikowane wiersze po czyszczeniu: 0
Liczba cech: 64


## 3. Najpierw test, potem każda operacja ucząca się na danych

Zbiór testowy (20%) pozostaje nietknięty do zakończenia doboru modeli, hiperparametrów i progu. `StratifiedKFold` zachowuje częstość bankructw w częściach treningowych.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "pozytywne:", int(y_train.sum()))
print("Test:", X_test.shape, "pozytywne:", int(y_test.sum()))

Train: (4680, 64) pozytywne: 326
Test: (1170, 64) pozytywne: 82


## 4. Przetwarzanie wewnątrz Pipeline

- `QuantileClipper` zapamiętuje percentyle tylko ze zbioru podanego do `fit` (w CV: wyłącznie fold treningowy).
- `SimpleImputer(strategy="median", add_indicator=True)` uzupełnia braki i dodaje automatycznie flagi brakujących danych.
- `RobustScaler` jest stosowany **tylko do regresji logistycznej**. Drzewa nie wymagają skalowania.
- W obecnym ARFF nie ma kategorycznych predyktorów. Gdyby pojawiły się w innych danych, obsłuży je `OneHotEncoder`.

W obu gałęziach `ColumnTransformer` dane są przetwarzane wyłącznie na podstawie treningu.

In [5]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Winsoryzacja każdej kolumny do percentyli obliczonych w fit()."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        a = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(a, self.lower, axis=0)
        self.upper_ = np.nanquantile(a, self.upper, axis=0)
        return self

    def transform(self, X):
        a = np.asarray(X, dtype=float)
        return np.clip(a, self.lower_, self.upper_)

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

def przygotowanie(skalowanie=False, winsoryzacja=True):
    kroki_num = []
    if winsoryzacja:
        kroki_num.append(("clip", QuantileClipper(0.01, 0.99)))
    kroki_num.append(("impute", SimpleImputer(strategy="median", add_indicator=True)))
    if skalowanie:
        kroki_num.append(("scale", RobustScaler()))

    transformatory = [("numeric", Pipeline(kroki_num), num_cols)]
    if cat_cols:
        transformatory.append(("categoric", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols))
    return ColumnTransformer(
        transformatory, remainder="drop", sparse_threshold=0.0
    )

def potok(model, skalowanie=False, winsoryzacja=True):
    return Pipeline([
        ("preprocess", przygotowanie(skalowanie, winsoryzacja)),
        ("model", model)
    ])

## 5. Baseline i 4 modele — porównanie 5-fold CV

Najważniejsza metryka porównawcza: `average_precision` (AP), zbliżona interpretacyjnie do pola pod krzywą Precision–Recall. Wartość AP dla klasyfikatora losowego/stałego jest zbliżona do częstości klasy pozytywnej. `ROC-AUC` jest miarą dodatkową.

Wyniki pochodzą wyłącznie z `X_train`; **nie dotykamy `X_test`**.

In [5]:

modele = {
    "Dummy (baseline)": potok(DummyClassifier(strategy="prior")),

    "HistGradientBoosting": potok(HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
        min_samples_leaf=30, l2_regularization=1.0,
        random_state=RANDOM_STATE
    )),

    "LogisticRegression": potok(LogisticRegression(
        C=0.2, class_weight="balanced", max_iter=3000,
        solver="lbfgs", random_state=RANDOM_STATE
    ), skalowanie=True),

    "RandomForest": potok(RandomForestClassifier(
        n_estimators=250, max_features="sqrt", min_samples_leaf=3,
        class_weight="balanced_subsample", n_jobs=-1,
        random_state=RANDOM_STATE
    )),

    "XGBoost": potok(XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        scale_pos_weight=1,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=2
    ))
}

# Walidacja krzyżowa dla wszystkich modeli
wyniki = []

for nazwa, estimator in modele.items():
    s = cross_validate(
        estimator, X_train, y_train,
        cv=cv,
        scoring={
            "ap": "average_precision",
            "roc": "roc_auc"
        },
        n_jobs=1,
        error_score="raise"
    )

    wyniki.append({
        "model": nazwa,
        "AP_mean": s["test_ap"].mean(),
        "AP_std": s["test_ap"].std(ddof=1),
        "ROC_AUC_mean": s["test_roc"].mean()
    })

wyniki_cv = pd.DataFrame(wyniki).sort_values(
    "AP_mean", ascending=False
)

print(wyniki_cv.to_string(index=False))
print("Przewaga nad baseline musi być oceniana w CV; nie na accuracy.")


               model  AP_mean   AP_std  ROC_AUC_mean
HistGradientBoosting 0.784081 0.038778      0.951521
             XGBoost 0.764439 0.049376      0.947008
  LogisticRegression 0.561264 0.075240      0.903502
        RandomForest 0.543421 0.058761      0.922973
    Dummy (baseline) 0.069658 0.000478      0.500000
Przewaga nad baseline musi być oceniana w CV; nie na accuracy.


## 6. Strojenie kandydata nr 1 (HGB)

To **przykładowa niewielka przestrzeń hiperparametrów**. W większym badaniu należy zwiększyć liczbę losowań i rozważyć walidację zewnętrzną/nested CV. `class_weight` jest strojonym wariantem, a nie obowiązkowym włączeniem.

In [6]:
hgb = modele["HistGradientBoosting"]
parametry = {
    "model__max_iter": [120, 200, 280],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_leaf_nodes": [7, 15, 31],
    "model__min_samples_leaf": [20, 40, 60],
    "model__l2_regularization": [0.0, 1.0, 10.0],
    "model__class_weight": [None, "balanced"]
}
search = RandomizedSearchCV(
    hgb, param_distributions=parametry,
    n_iter=25, scoring="average_precision",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    refit=True, random_state=RANDOM_STATE, n_jobs=1,
    error_score="raise", return_train_score=False
)
search.fit(X_train, y_train)
print("Najlepsze AP w CV:", round(search.best_score_, 4))
print("Wybrane parametry:", search.best_params_)
najlepszy = search.best_estimator_

Najlepsze AP w CV: 0.7763
Wybrane parametry: {'model__min_samples_leaf': 20, 'model__max_leaf_nodes': 15, 'model__max_iter': 120, 'model__learning_rate': 0.05, 'model__l2_regularization': 0.0, 'model__class_weight': None}


In [7]:

# Strojenie hiperparametrów XGBoost

parametry_xgb = {
    "model__n_estimators": [150, 250, 400],
    "model__learning_rate": [0.02, 0.05, 0.1],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.7, 0.85, 1.0],
    "model__reg_lambda": [1, 5, 10],
    "model__scale_pos_weight": [1, 3, 7, 13]
}

cv_inner = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

search_xgb = RandomizedSearchCV(
    estimator=modele["XGBoost"],
    param_distributions=parametry_xgb,
    n_iter=25,
    scoring="average_precision",
    cv=cv_inner,
    refit=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
    error_score="raise"
)

search_xgb.fit(X_train, y_train)

print("Najlepszy wynik AP:", search_xgb.best_score_)
print("Najlepsze parametry:")
print(search_xgb.best_params_)


Najlepszy wynik AP: 0.762379741379411
Najlepsze parametry:
{'model__subsample': 0.7, 'model__scale_pos_weight': 7, 'model__reg_lambda': 1, 'model__n_estimators': 150, 'model__min_child_weight': 3, 'model__max_depth': 5, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.85}


In [8]:

cv_outer = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

for nazwa, procedura in [
    ("HistGradientBoosting", search),
    ("XGBoost", search_xgb)
]:
    ocena = cross_validate(
        procedura,
        X_train,
        y_train,
        cv=cv_outer,
        scoring="average_precision",
        n_jobs=1
    )

    print(
        nazwa,
        "AP:", round(ocena["test_score"].mean(), 4),
        "STD:", round(ocena["test_score"].std(ddof=1), 4)
    )


HistGradientBoosting AP: 0.7896 STD: 0.0208
XGBoost AP: 0.7796 STD: 0.0238


## 7. Wybór progu wyłącznie na treningu

Próg 0,5 często nie odpowiada celowi wykrywania bankructw. Dla demonstracji wybieramy próg maksymalizujący **F2** (większy nacisk na czułość niż na precyzję) na predykcjach out-of-fold. Jeśli znasz koszty fałszywego alarmu i niewykrytego bankructwa, wybierz próg według tych kosztów zamiast F2.

In [9]:
p_oof = cross_val_predict(
    najlepszy, X_train, y_train, cv=cv,
    method="predict_proba", n_jobs=1
)[:, 1]
precyzja, czulosc, progi = precision_recall_curve(y_train, p_oof)
wynik_f2 = (5 * precyzja[:-1] * czulosc[:-1]) / (
    4 * precyzja[:-1] + czulosc[:-1] + 1e-12
)
idx = int(np.argmax(wynik_f2))
prog = float(progi[idx])
print("Próg (F2, out-of-fold):", round(prog, 5))
print("F2 na OOF:", round(float(wynik_f2[idx]), 4))
# Uwaga: do rygorystycznej estymacji skuteczności doboru progu użyj nested CV.
# Tu niezależny test pozostaje nietknięty i zostanie użyty tylko raz.

Próg (F2, out-of-fold): 0.08113
F2 na OOF: 0.7229


## 8. Jednorazowy test końcowy

Nie zmieniaj progu ani hiperparametrów na podstawie wyników na `X_test`: spowodowałoby to dalszą optymalizację pod test.

In [10]:
# RandomizedSearchCV(refit=True) już dopasował model na całym X_train.
p_test = najlepszy.predict_proba(X_test)[:, 1]
pred_test = (p_test >= prog).astype(int)

metryki_test = {
    "AP": average_precision_score(y_test, p_test),
    "ROC_AUC": roc_auc_score(y_test, p_test),
    "Precision": precision_score(y_test, pred_test, zero_division=0),
    "Recall": recall_score(y_test, pred_test, zero_division=0),
    "F2": fbeta_score(y_test, pred_test, beta=2, zero_division=0),
    "Balanced_accuracy": balanced_accuracy_score(y_test, pred_test),
    "Brier_score": brier_score_loss(y_test, p_test)
}
print(pd.Series(metryki_test).round(4))
print("Macierz pomyłek [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, pred_test))
print(classification_report(y_test, pred_test, digits=3, zero_division=0))

AP                   0.8474
ROC_AUC              0.9678
Precision            0.5224
Recall               0.8537
F2                   0.7576
Balanced_accuracy    0.8974
Brier_score          0.0225
dtype: float64
Macierz pomyłek [[TN, FP], [FN, TP]]:
[[1024   64]
 [  12   70]]
              precision    recall  f1-score   support

           0      0.988     0.941     0.964      1088
           1      0.522     0.854     0.648        82

    accuracy                          0.935      1170
   macro avg      0.755     0.897     0.806      1170
weighted avg      0.956     0.935     0.942      1170



## 9. Analizy wrażliwości (wykonać jako oddzielne eksperymenty na treningu)

1. HGB bez winsoryzacji i / lub bez flag braków; ewentualnie HGB przyjmujący NaN bez imputacji.
2. Porównanie `class_weight=None` vs `"balanced"`.
3. Usunięcie `Attr37` (~43% braków), wyłącznie jako ablation study; nie usuwaj automatycznie.
4. PCA / Elastic Net jako **alternatywy** dla modelu liniowego, dopasowane wewnątrz CV.
5. Kalibracja prawdopodobieństw na wydzielonej części treningowej (np. `CalibratedClassifierCV`) i diagnostyka wykresu kalibracyjnego.
6. Walidacja firmowa (`GroupKFold`) i/lub chronologiczna, jeśli uzyskasz identyfikatory przedsiębiorstw i daty. Bez tych metadanych możliwy pozostaje ukryty wyciek i optymistyczna ocena.

**Nie używaj SMOTE przed podziałem**, a w pierwszej kolejności sprawdź wagi klas oraz dobór progu.

**Interpretacja:** po ustaleniu modelu można obliczyć `permutation_importance` na niezależnym zestawie interpretacyjnym albo SHAP (poza podstawowym scikit-learn) i zweryfikować sens ekonomiczny najważniejszych wskaźników.

## 10. Utrwalenie końcowego modelu

Zapisuje dopasowaną całość: winsoryzację, imputację, skalowanie/kodowanie (jeśli dotyczy) oraz estymator. W zastosowaniach produkcyjnych dołącz wersję bibliotek i opis pochodzenia danych.

In [16]:
import joblib
joblib.dump({"pipeline": najlepszy, "threshold": prog,
             "feature_names": list(X.columns)},
            "model_bankructwo_5year.joblib")
print("Zapisano model_bankructwo_5year.joblib")

Zapisano model_bankructwo_5year.joblib


### Test

In [17]:
import joblib
import pandas as pd

# Wczytanie zapisanego modelu
zapis = joblib.load("model_bankructwo_5year.joblib")

model = zapis["pipeline"]
prog = zapis["threshold"]
cechy = zapis["feature_names"]

# Przykładowa predykcja na jednym przedsiębiorstwie ze zbioru testowego
firma = X_test.iloc[[0]][cechy]

prawdopodobienstwo = model.predict_proba(firma)[0, 1]
predykcja = int(prawdopodobienstwo >= prog)

print("Liczba wymaganych wskaźników:", len(cechy))
print("Próg decyzyjny:", round(prog, 5))
print("Wynik modelu:", round(prawdopodobienstwo, 4))
print("Predykcja:", predykcja)

Liczba wymaganych wskaźników: 64
Próg decyzyjny: 0.08113
Wynik modelu: 0.0017
Predykcja: 0


In [6]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# Te same podziały CV dla wszystkich wariantów
cv_test = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Używamy wybranego wcześniej HGB, jeśli jest dostępny
if "najlepszy" in globals() and isinstance(najlepszy, Pipeline):
    model_hgb = clone(najlepszy.named_steps["model"])
else:
    model_hgb = HistGradientBoostingClassifier(random_state=42)

# Warianty danych
eksperymenty = {
    "64 wskaźniki": X_train,
    "Bez Attr21": X_train.drop(columns=["Attr21"]),
    "Bez Attr21 i Attr27": X_train.drop(
        columns=["Attr21", "Attr27"]
    ),
    "Tylko flagi braków (64)": X_train.isna().astype(int),
    "Tylko flagi Attr21 i Attr27": (
        X_train[["Attr21", "Attr27"]].isna().astype(int)
    )
}

wyniki = []
wyniki_foldy = []

for nazwa, dane in eksperymenty.items():

    print(f"Trwa eksperyment: {nazwa}")

    if nazwa.startswith("Tylko flagi"):
        # Dane zawierają wyłącznie 0 i 1
        pipeline = clone(model_hgb)

    else:
        # Taki sam preprocessing jak w naszym modelu
        pipeline = Pipeline([
            ("clip", QuantileClipper(0.01, 0.99)),
            ("impute", SimpleImputer(
                strategy="median",
                add_indicator=True
            )),
            ("model", clone(model_hgb))
        ])

    scores = cross_validate(
        pipeline,
        dane,
        y_train,
        cv=cv_test,
        scoring={
            "AP": "average_precision",
            "ROC_AUC": "roc_auc"
        },
        n_jobs=-1
    )

    ap = scores["test_AP"]
    roc = scores["test_ROC_AUC"]

    wyniki.append({
        "Wariant": nazwa,
        "AP średnie": ap.mean(),
        "AP odch. std.": ap.std(),
        "ROC AUC średnie": roc.mean(),
        "ROC AUC odch. std.": roc.std()
    })

    for i, (ap_fold, roc_fold) in enumerate(zip(ap, roc), 1):
        wyniki_foldy.append({
            "Wariant": nazwa,
            "Fold": i,
            "AP": ap_fold,
            "ROC_AUC": roc_fold
        })

wyniki_df = (
    pd.DataFrame(wyniki)
    .sort_values("AP średnie", ascending=False)
    .reset_index(drop=True)
)

foldy_df = pd.DataFrame(wyniki_foldy)

display(wyniki_df.round(4))

Trwa eksperyment: 64 wskaźniki
Trwa eksperyment: Bez Attr21
Trwa eksperyment: Bez Attr21 i Attr27
Trwa eksperyment: Tylko flagi braków (64)
Trwa eksperyment: Tylko flagi Attr21 i Attr27


,Wariant,AP średnie,AP odch. std.,ROC AUC średnie,ROC AUC odch. std.
0,64 wskaźniki,0.7920,0.0336,0.9535,0.0113
1,Bez Attr21,0.7219,0.0370,0.9543,0.0074
2,Bez Attr21 i Attr27,0.5363,0.0543,0.9005,0.0171
3,Tylko flagi braków (64),0.4413,0.0679,0.7313,0.0359
4,Tylko flagi Attr21 i Attr27,0.3760,0.0785,0.7270,0.0402


In [7]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate
import pandas as pd

warianty = {
    "A: mediana + flagi": Pipeline([
        ("clip", QuantileClipper()),
        ("impute", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("model", clone(model_hgb))
    ]),

    "B: mediana bez flag": Pipeline([
        ("clip", QuantileClipper()),
        ("impute", SimpleImputer(
            strategy="median",
            add_indicator=False
        )),
        ("model", clone(model_hgb))
    ]),

    "C: natywne NaN": Pipeline([
        ("clip", QuantileClipper()),
        ("model", clone(model_hgb))
    ])
}

wyniki_brakow = []

for nazwa, pipeline in warianty.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv_test,
        scoring={
            "AP": "average_precision",
            "ROC_AUC": "roc_auc"
        },
        n_jobs=-1
    )

    wyniki_brakow.append({
        "Wariant": nazwa,
        "AP": scores["test_AP"].mean(),
        "AP std": scores["test_AP"].std(),
        "ROC AUC": scores["test_ROC_AUC"].mean()
    })

wyniki_brakow_df = pd.DataFrame(wyniki_brakow)

display(wyniki_brakow_df.round(4))

,Wariant,AP,AP std,ROC AUC
0,A: mediana + flagi,0.7920,0.0336,0.9535
1,B: mediana bez flag,0.7543,0.0375,0.9466
2,C: natywne NaN,0.7756,0.0503,0.9495


In [9]:
model_sama_mediana = Pipeline([
    ("clip", QuantileClipper(0.01, 0.99)),
    ("impute", SimpleImputer(
        strategy="median",
        add_indicator=False
    )),
    ("model", clone(model_hgb))
])

model_sama_mediana.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('clip', ...), ('impute', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,lower,0.01
,upper,0.99
Name,Type,Value
lower_,"ndarray[float64](64,)","[-0.64, 0.02,-1.18,..., 5.37, 0.49, 0.18]"
upper_,"ndarray[float64](64,)","[ 0.53, 2.16, 0.91,...,737.27, 48.67,423.37]"


In [10]:
from sklearn.model_selection import cross_validate
from sklearn.base import clone
import numpy as np
import pandas as pd

wyniki_cv = cross_validate(
    clone(model_sama_mediana),
    X_train,
    y_train,
    cv=cv_test,
    scoring={
        "AP": "average_precision",
        "ROC_AUC": "roc_auc"
    },
    n_jobs=-1
)

print("MODEL: HGB + MEDIANA BEZ FLAG")
print("-" * 40)

print("Average Precision (AP):",
      round(wyniki_cv["test_AP"].mean(), 4))

print("Odchylenie standardowe AP:",
      round(wyniki_cv["test_AP"].std(), 4))

print("ROC AUC:",
      round(wyniki_cv["test_ROC_AUC"].mean(), 4))

MODEL: HGB + MEDIANA BEZ FLAG
----------------------------------------
Average Precision (AP): 0.7543
Odchylenie standardowe AP: 0.0375
ROC AUC: 0.9466


In [12]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, accuracy_score

cechy = X_train.columns.drop(["Attr21", "Attr27"])

model_62 = Pipeline([
    ("clip", QuantileClipper(0.01, 0.99)),
    ("impute", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("model", clone(model_hgb))
])

model_62.fit(X_train[cechy], y_train)

p = model_62.predict_proba(X_test[cechy])[:, 1]
pred = (p >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

print("Poprawnie wykryte bankructwa:", tp, "z", tp + fn)
print("Przeoczone bankructwa:", fn)
print("Poprawnie rozpoznane pozostałe firmy:", tn, "z", tn + fp)
print("Fałszywe alarmy:", fp)
print("Łącznie poprawne decyzje:", tp + tn, "z", len(y_test))
print("Accuracy:", round(accuracy_score(y_test, pred) * 100, 2), "%")

Poprawnie wykryte bankructwa: 35 z 82
Przeoczone bankructwa: 47
Poprawnie rozpoznane pozostałe firmy: 1067 z 1088
Fałszywe alarmy: 21
Łącznie poprawne decyzje: 1102 z 1170
Accuracy: 94.19 %


In [13]:
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    precision_recall_curve,
    fbeta_score,
    precision_score,
    recall_score
)

# Usuwamy Attr21 i Attr27
cechy_62 = X_train.columns.drop(["Attr21", "Attr27"])
X_train_62 = X_train[cechy_62]

# 5-krotna walidacja krzyżowa
cv_62 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Predykcje OOF — każda firma oceniana przez model,
# który nie trenował na tej firmie
p_oof = cross_val_predict(
    clone(model_62),
    X_train_62,
    y_train,
    cv=cv_62,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Wyznaczenie precyzji i czułości dla różnych progów
precision, recall, thresholds = precision_recall_curve(
    y_train,
    p_oof
)

# Obliczenie F2 dla każdego progu
f2 = (
    5 * precision[:-1] * recall[:-1]
    / (4 * precision[:-1] + recall[:-1] + 1e-12)
)

# Wybór progu z najwyższym F2
indeks = np.argmax(f2)
najlepszy_prog_62 = float(thresholds[indeks])

print("NAJLEPSZY PRÓG:", round(najlepszy_prog_62, 5))
print("F2:", round(f2[indeks], 4))
print("Precision:", round(precision[indeks], 4))
print("Recall:", round(recall[indeks], 4))

NAJLEPSZY PRÓG: 0.02119
F2: 0.6142
Precision: 0.3663
Recall: 0.7393


In [14]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    average_precision_score,
    roc_auc_score
)

X_test_62 = X_test[cechy_62]

# Trenujemy model na całym zbiorze treningowym
model_62_final = clone(model_62)
model_62_final.fit(X_train_62, y_train)

# Wyniki ryzyka dla danych testowych
p_test = model_62_final.predict_proba(X_test_62)[:, 1]

print("AP:", round(average_precision_score(y_test, p_test), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, p_test), 4))

for nazwa, prog in [
    ("Standardowy próg", 0.5),
    ("Próg dobrany F2", najlepszy_prog_62)
]:
    pred = (p_test >= prog).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, pred, labels=[0, 1]
    ).ravel()

    print("\n" + "=" * 45)
    print(nazwa, "=", round(prog, 5))
    print("=" * 45)

    print("Wykryte bankructwa:", tp, "z", tp + fn)
    print("Przeoczone bankructwa:", fn)
    print("Fałszywe alarmy:", fp)
    print("Poprawnie rozpoznane pozostałe firmy:", tn)

    print("Recall:", round(recall_score(y_test, pred), 4))
    print("Precision:", round(precision_score(
        y_test, pred, zero_division=0
    ), 4))
    print("F2:", round(fbeta_score(
        y_test, pred, beta=2, zero_division=0
    ), 4))
    print("Accuracy:", round(
        accuracy_score(y_test, pred) * 100, 2
    ), "%")

AP: 0.5624
ROC-AUC: 0.8912

Standardowy próg = 0.5
Wykryte bankructwa: 35 z 82
Przeoczone bankructwa: 47
Fałszywe alarmy: 21
Poprawnie rozpoznane pozostałe firmy: 1067
Recall: 0.4268
Precision: 0.625
F2: 0.4557
Accuracy: 94.19 %

Próg dobrany F2 = 0.02119
Wykryte bankructwa: 61 z 82
Przeoczone bankructwa: 21
Fałszywe alarmy: 149
Poprawnie rozpoznane pozostałe firmy: 939
Recall: 0.7439
Precision: 0.2905
F2: 0.5669
Accuracy: 85.47 %


In [15]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate

# 12 wskaźników z Twojej tabeli
cechy_12 = [
    "Attr1", "Attr2", "Attr3", "Attr5",
    "Attr6", "Attr24", "Attr29", "Attr35",
    "Attr46", "Attr56", "Attr58", "Attr62"
]

X_train_12 = X_train[cechy_12]
X_test_12 = X_test[cechy_12]

warianty_12 = {
    "12: mediana + flagi": True,
    "12: sama mediana": False
}

wyniki_12 = []
modele_12 = {}

for nazwa, flagi in warianty_12.items():

    pipeline = Pipeline([
        ("clip", QuantileClipper(0.01, 0.99)),
        ("impute", SimpleImputer(
            strategy="median",
            add_indicator=flagi
        )),
        ("model", clone(model_hgb))
    ])

    modele_12[nazwa] = pipeline

    scores = cross_validate(
        pipeline,
        X_train_12,
        y_train,
        cv=cv_test,
        scoring={
            "AP": "average_precision",
            "ROC_AUC": "roc_auc"
        },
        n_jobs=-1
    )

    wyniki_12.append({
        "Wariant": nazwa,
        "AP": scores["test_AP"].mean(),
        "AP std": scores["test_AP"].std(),
        "ROC AUC": scores["test_ROC_AUC"].mean()
    })

wyniki_12_df = (
    pd.DataFrame(wyniki_12)
    .sort_values("AP", ascending=False)
    .reset_index(drop=True)
)

display(wyniki_12_df.round(4))

,Wariant,AP,AP std,ROC AUC
0,12: sama mediana,0.5555,0.0515,0.8814
1,12: mediana + flagi,0.5478,0.0499,0.8804


In [16]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    recall_score,
    precision_score,
    fbeta_score,
    accuracy_score
)

# Wybór wariantu według AP na treningu
nazwa_12 = wyniki_12_df.iloc[0]["Wariant"]
model_12 = clone(modele_12[nazwa_12])

# Predykcje poza próbą treningową (OOF)
p_oof_12 = cross_val_predict(
    clone(model_12),
    X_train_12,
    y_train,
    cv=cv_test,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Dobór progu maksymalizującego F2
precision, recall, thresholds = precision_recall_curve(
    y_train, p_oof_12
)

f2 = (
    5 * precision[:-1] * recall[:-1]
    / (4 * precision[:-1] + recall[:-1] + 1e-12)
)

najlepszy_prog_12 = float(thresholds[np.argmax(f2)])

# Trening na całym zbiorze treningowym
model_12.fit(X_train_12, y_train)

# Predykcje testowe
p_test_12 = model_12.predict_proba(X_test_12)[:, 1]
pred_12 = (p_test_12 >= najlepszy_prog_12).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test, pred_12, labels=[0, 1]
).ravel()

print("=" * 45)
print("MODEL HGB — 12 WSKAŹNIKÓW")
print("=" * 45)
print("Wybrany wariant:", nazwa_12)
print("Próg F2:", round(najlepszy_prog_12, 5))

print("\nAP:", round(
    average_precision_score(y_test, p_test_12), 4
))
print("ROC AUC:", round(
    roc_auc_score(y_test, p_test_12), 4
))

print("\nWykryte bankructwa:", tp, "z", tp + fn)
print("Przeoczone bankructwa:", fn)
print("Fałszywe alarmy:", fp)
print("Poprawnie rozpoznane pozostałe firmy:", tn)

print("\nRecall:", round(recall_score(y_test, pred_12), 4))
print("Precision:", round(
    precision_score(y_test, pred_12, zero_division=0), 4
))
print("F2:", round(
    fbeta_score(y_test, pred_12, beta=2), 4
))
print("Accuracy:", round(
    accuracy_score(y_test, pred_12) * 100, 2
), "%")

MODEL HGB — 12 WSKAŹNIKÓW
Wybrany wariant: 12: sama mediana
Próg F2: 0.02887

AP: 0.5498
ROC AUC: 0.8961

Wykryte bankructwa: 65 z 82
Przeoczone bankructwa: 17
Fałszywe alarmy: 155
Poprawnie rozpoznane pozostałe firmy: 933

Recall: 0.7927
Precision: 0.2955
F2: 0.5931
Accuracy: 85.3 %


In [17]:
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate, StratifiedKFold

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# 12 wybranych wskaźników
cechy_12 = [
    "Attr1", "Attr2", "Attr3", "Attr5",
    "Attr6", "Attr24", "Attr29", "Attr35",
    "Attr46", "Attr56", "Attr58", "Attr62"
]

X_12 = X_train[cechy_12]

# Modele do porównania
modele = {
    "HGB": clone(model_hgb),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=1
    ),

    "Regresja logistyczna": LogisticRegression(
        max_iter=3000,
        random_state=42
    )
}

cv_porownanie = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

wyniki_modeli = []

for nazwa, model in modele.items():

    kroki = [
        ("clip", QuantileClipper(0.01, 0.99)),
        ("impute", SimpleImputer(
            strategy="median",
            add_indicator=True
        ))
    ]

    # Regresja logistyczna wymaga skalowania
    if nazwa == "Regresja logistyczna":
        kroki.append(("scale", RobustScaler()))

    kroki.append(("model", model))

    pipeline = Pipeline(kroki)

    scores = cross_validate(
        pipeline,
        X_12,
        y_train,
        cv=cv_porownanie,
        scoring={
            "AP": "average_precision",
            "ROC_AUC": "roc_auc"
        },
        n_jobs=-1
    )

    wyniki_modeli.append({
        "Model": nazwa,
        "AP": scores["test_AP"].mean(),
        "AP std": scores["test_AP"].std(),
        "ROC AUC": scores["test_ROC_AUC"].mean()
    })

wyniki_modeli_df = (
    pd.DataFrame(wyniki_modeli)
    .sort_values("AP", ascending=False)
    .reset_index(drop=True)
)

display(wyniki_modeli_df.round(4))

,Model,AP,AP std,ROC AUC
0,XGBoost,0.5701,0.0296,0.8808
1,HGB,0.5478,0.0499,0.8804
2,Random Forest,0.4942,0.0291,0.8825
3,Extra Trees,0.4412,0.0222,0.8671
4,Regresja logistyczna,0.4158,0.0389,0.8173


In [18]:
import numpy as np

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    fbeta_score
)

cechy_12 = [
    "Attr1", "Attr2", "Attr3", "Attr5",
    "Attr6", "Attr24", "Attr29", "Attr35",
    "Attr46", "Attr56", "Attr58", "Attr62"
]

X_tr = X_train[cechy_12]
X_te = X_test[cechy_12]

model_xgb_12 = Pipeline([
    ("clip", QuantileClipper(0.01, 0.99)),
    ("impute", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    ))
])

cv_xgb = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Dobór progu WYŁĄCZNIE na danych treningowych
p_oof = cross_val_predict(
    model_xgb_12,
    X_tr,
    y_train,
    cv=cv_xgb,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_train, p_oof
)

f2 = (
    5 * precision[:-1] * recall[:-1]
    / (4 * precision[:-1] + recall[:-1] + 1e-12)
)

prog_xgb_12 = float(thresholds[np.argmax(f2)])

# Trening na całym zbiorze treningowym
model_xgb_12.fit(X_tr, y_train)

# Ocena na zbiorze testowym
p_test = model_xgb_12.predict_proba(X_te)[:, 1]
pred = (p_test >= prog_xgb_12).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test, pred, labels=[0, 1]
).ravel()

print("XGBOOST — 12 WSKAŹNIKÓW")
print("=" * 45)
print("Próg F2:", round(prog_xgb_12, 5))
print("Wykryte bankructwa:", tp, "z", tp + fn)
print("Przeoczone bankructwa:", fn)
print("Fałszywe alarmy:", fp)
print("Poprawnie rozpoznane pozostałe firmy:", tn)

print("\nPrecision:", round(precision_score(y_test, pred), 4))
print("Recall:", round(recall_score(y_test, pred), 4))
print("F2:", round(fbeta_score(y_test, pred, beta=2), 4))
print("AP:", round(average_precision_score(y_test, p_test), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, p_test), 4))

XGBOOST — 12 WSKAŹNIKÓW
Próg F2: 0.12463
Wykryte bankructwa: 57 z 82
Przeoczone bankructwa: 25
Fałszywe alarmy: 87
Poprawnie rozpoznane pozostałe firmy: 1001

Precision: 0.3958
Recall: 0.6951
F2: 0.6038
AP: 0.5469
ROC-AUC: 0.8981


In [22]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

# Usuwamy tylko dwa podejrzane wskaźniki
cechy_62 = X_train.columns.drop(["Attr21", "Attr27"])

X_train_62 = X_train[cechy_62]
X_test_62 = X_test[cechy_62]

# XGBoost sam obsługuje NaN
model_xgb_nan = Pipeline([
    ("clip", QuantileClipper(0.01, 0.99)),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=1
    ))
])

cv_nan = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

wyniki_nan = cross_validate(
    model_xgb_nan,
    X_train_62,
    y_train,
    cv=cv_nan,
    scoring={
        "AP": "average_precision",
        "ROC_AUC": "roc_auc"
    },
    n_jobs=-1
)

print("Liczba wskaźników:", X_train_62.shape[1])
print("AP:", round(wyniki_nan["test_AP"].mean(), 4))
print("ROC-AUC:", round(wyniki_nan["test_ROC_AUC"].mean(), 4))

Liczba wskaźników: 62
AP: 0.5254
ROC-AUC: 0.8871


In [23]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    precision_recall_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    fbeta_score,
    average_precision_score,
    roc_auc_score
)

# Dobór progu na danych treningowych
p_oof = cross_val_predict(
    clone(model_xgb_nan),
    X_train_62,
    y_train,
    cv=cv_nan,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_train, p_oof
)

f2 = (
    5 * precision[:-1] * recall[:-1]
    / (4 * precision[:-1] + recall[:-1] + 1e-12)
)

prog_xgb_nan = float(thresholds[np.argmax(f2)])

# Trening końcowy
model_xgb_nan.fit(X_train_62, y_train)

p_test = model_xgb_nan.predict_proba(X_test_62)[:, 1]
pred = (p_test >= prog_xgb_nan).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test, pred, labels=[0, 1]
).ravel()

print("XGBOOST — 62 WSKAŹNIKI, NATYWNE NaN")
print("=" * 45)
print("Próg F2:", round(prog_xgb_nan, 5))
print("Wykryte bankructwa:", tp, "z", tp + fn)
print("Przeoczone bankructwa:", fn)
print("Fałszywe alarmy:", fp)
print("Poprawnie rozpoznane pozostałe firmy:", tn)

print("Precision:", round(precision_score(y_test, pred), 4))
print("Recall:", round(recall_score(y_test, pred), 4))
print("F2:", round(fbeta_score(y_test, pred, beta=2), 4))
print("AP:", round(average_precision_score(y_test, p_test), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, p_test), 4))

XGBOOST — 62 WSKAŹNIKI, NATYWNE NaN
Próg F2: 0.06524
Wykryte bankructwa: 65 z 82
Przeoczone bankructwa: 17
Fałszywe alarmy: 169
Poprawnie rozpoznane pozostałe firmy: 919
Precision: 0.2778
Recall: 0.7927
F2: 0.5783
AP: 0.5773
ROC-AUC: 0.9016


In [24]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, pred)

print("Accuracy:", round(accuracy, 4))
print("Accuracy (%):", round(accuracy * 100, 2), "%")
print("Poprawnie sklasyfikowane firmy:",
      (pred == y_test).sum(), "z", len(y_test))

Accuracy: 0.841
Accuracy (%): 84.1 %
Poprawnie sklasyfikowane firmy: 984 z 1170
